In [5]:
import pandas as pd
import numpy as np
import glob
import os
import re

# --- Configuration ---
DATA_DIR = './simulation_Lorenz96'       # Directory containing your result files
FILE_PATTERN = '*.csv'        # File pattern to match
TARGET_DIMENSION = 50         # Set to 10 or 50

# Output filename for the best configurations
OUTPUT_CSV_MIN_LOSS = 'best_performance.csv' 

# The Master List of Series (Order matters!)
SERIES_NAMES = [
    'D10_T500_F10', 'D10_T500_F40', 
    'D10_T1000_F10', 'D10_T1000_F40', 
    'D10_T2000_F10', 'D10_T2000_F40', 
    'D50_T500_F10', 'D50_T500_F40', 
    'D50_T1000_F10', 'D50_T1000_F40', 
    'D50_T2000_F10', 'D50_T2000_F40'
]

# Method Display Names for the Table
METHOD_DISPLAY_NAMES = {
    'Jacob_L1': 'Jacob-L1',
    'Jacob_F': 'Jacob-F',
    'Shapley': 'Shapley',
    'Fast_Shap': 'F-Shap'
}
METHOD_ORDER = ['Jacob_L1', 'Jacob_F', 'Shapley', 'Fast_Shap']

def parse_filename_and_params(filepath):
    basename = os.path.basename(filepath)
    name_no_ext = os.path.splitext(basename)[0]
    parts = name_no_ext.split('_')
    
    if len(parts) < 6: return None
    
    try:
        series_id = int(parts[2])
    except ValueError:
        return None 

    list_index = series_id - 1
    if list_index < 0 or list_index >= len(SERIES_NAMES):
        return None 

    series_str = SERIES_NAMES[list_index]
    match = re.search(r'D(\d+)_T(\d+)_F(\d+)', series_str)
    if not match: return None
    
    params = {'D': int(match.group(1)), 'T': int(match.group(2)), 'F': int(match.group(3))}

    return {
        'series_str': series_str,
        'subject': parts[3],
        'model': parts[4],
        'method': '_'.join(parts[5:]),
        'filename': basename,
        **params 
    }

def process_files(data_dir, pattern):
    files = glob.glob(os.path.join(data_dir, pattern))
    rows_min_loss = []   
    
    print(f"Processing {len(files)} files...")

    for f in files:
        meta = parse_filename_and_params(f)
        if not meta: continue
        
        try:
            df = pd.read_csv(f)
            
            # Ensure metrics are numeric
            cols_to_check = ['val_loss', 'AUROC', 'AUPRC']
            for c in cols_to_check:
                if c in df.columns:
                    df[c] = pd.to_numeric(df[c], errors='coerce')
            
            if df.empty or 'val_loss' not in df.columns: 
                continue

            # --- STRICT SELECTION: MIN VAL LOSS ---
            # Sort by val_loss ascending and pick the very first row.
            best_candidate = df.sort_values(by='val_loss', ascending=True).iloc[0]
            
            # Store results
            row_data = best_candidate.to_dict()
            row_data.update(meta) 
            rows_min_loss.append(row_data)

        except Exception as e:
            print(f"Error processing {os.path.basename(f)}: {e}")
            continue

    return pd.DataFrame(rows_min_loss)

def save_best_to_csv(df, filename):
    """
    Saves the dataframe of selected best runs to CSV.
    Organizes columns so important identifiers come first.
    """
    if df.empty:
        print(f"No data to save for {filename}")
        return

    # 1. Define column order priorities
    meta_cols = ['method', 'D', 'T', 'F', 'subject', 'series_str', 'filename']
    metric_cols = ['AUROC', 'AUPRC', 'val_loss', 'epoch']
    
    # 2. Filter for columns that actually exist
    meta_cols = [c for c in meta_cols if c in df.columns]
    metric_cols = [c for c in metric_cols if c in df.columns]
    
    # 3. Get everything else (hyperparams etc.)
    other_cols = [c for c in df.columns if c not in meta_cols and c not in metric_cols]
    
    # 4. Construct final column list
    final_cols = meta_cols + metric_cols + other_cols
    
    # 5. Reorder and Save
    df_sorted = df[final_cols]
    # print(f"Saving {len(df)} 'Best Config' rows to {filename}...")
    df_sorted.to_csv(filename, index=False)

def get_aggregated_stats(df_raw):
    """Aggregates for the LaTeX table"""
    if df_raw.empty: return pd.DataFrame()
    return df_raw.groupby(['D', 'F', 'T', 'method']).agg(
        auroc_mean=('AUROC', 'mean'),
        auroc_std=('AUROC', 'std'),
        auprc_mean=('AUPRC', 'mean'),
        auprc_std=('AUPRC', 'std')
    ).reset_index()

def generate_latex_table(df, target_d, strategy_name):
    df_sub = df[df['D'] == target_d].copy()
    if df_sub.empty:
        print(f"No data found for Dimension D={target_d}")
        return

    unique_Fs = sorted(df_sub['F'].unique())
    unique_Ts = sorted(df_sub['T'].unique())
    if not unique_Fs or not unique_Ts: return

    # Header
    header_top = "        &  "
    header_mid = ""
    col_counter = 3
    for f_val in unique_Fs:
        header_top += f"& \\multicolumn{{{len(unique_Ts)}}}{{c}}{{F = {f_val}}} "
        header_mid += f"\\cmidrule(lr){{{col_counter}-{col_counter + len(unique_Ts) - 1}}} "
        col_counter += len(unique_Ts)
        
    header_bot = "        Penalty & Metric"
    for _ in unique_Fs:
        for t_val in unique_Ts:
            header_bot += f" & T={t_val}"

    latex_header = f"{header_top} \\\\\n        {header_mid}\n{header_bot} \\\\"

    # Body
    rows = []
    for method in METHOD_ORDER:
        disp = METHOD_DISPLAY_NAMES.get(method, method)
        line_roc = f"        \\multirow{{2}}{{*}}{{{disp}}}\n          & AUROC"
        line_prc = "          & AUPRC"
        
        for f_val in unique_Fs:
            for t_val in unique_Ts:
                item = df_sub[(df_sub['method'] == method) & (df_sub['F'] == f_val) & (df_sub['T'] == t_val)]
                
                if not item.empty:
                    m_roc = item.iloc[0]['auroc_mean']
                    s_roc = item.iloc[0]['auroc_std']
                    m_prc = item.iloc[0]['auprc_mean']
                    s_prc = item.iloc[0]['auprc_std']
                    line_roc += f" & \\res{{{m_roc:.3f}}}{{{s_roc:.3f}}}"
                    line_prc += f" & \\res{{{m_prc:.3f}}}{{{s_prc:.3f}}}"
                else:
                    line_roc += " & --"
                    line_prc += " & --"
        rows.append(f"{line_roc} \\\\\n          \\addlinespace[0.5em]\n{line_prc} \\\\")

    latex_body = "\n        \\midrule\n        ".join(rows)
    total_cols = 2 + (len(unique_Fs) * len(unique_Ts))
    col_def = "ll" + "c" * (total_cols - 2)

    print(r"""
% -----------------------------------------------------------------------
% TABLE: """ + strategy_name + r"""
% -----------------------------------------------------------------------
\begin{table}[ht]
    \centering
    \caption{Performance comparison on Lorenz-96 ($d=""" + str(target_d) + r"""$). Strategy: """ + strategy_name + r"""}
    \label{tab:lorenz96_""" + str(target_d) + r"""_""" + strategy_name.replace(" ", "_") + r"""}
    
    \newcommand{\res}[2]{\shortstack{#1 \\ {\scriptsize (#2)}}}
    
    \resizebox{\columnwidth}{!}{%
    \begin{tabular}{""" + col_def + r"""}
        \toprule
""" + latex_header + r"""
        \midrule
""" + latex_body + r"""
        \bottomrule
    \end{tabular}%
    }
\end{table}
""")

# --- Main Execution ---

df_results = process_files(DATA_DIR, FILE_PATTERN)

# 2. SAVE the best configs to CSV
# save_best_to_csv(df_results, OUTPUT_CSV_MIN_LOSS)

# 3. Aggregate stats
stats_agg = get_aggregated_stats(df_results)

# 4. Print Table
if not stats_agg.empty:
    generate_latex_table(stats_agg, target_d=TARGET_DIMENSION, strategy_name="Best performance")

Processing 80 files...

% -----------------------------------------------------------------------
% TABLE: Best performance
% -----------------------------------------------------------------------
\begin{table}[ht]
    \centering
    \caption{Performance comparison on Lorenz-96 ($d=50$). Strategy: Best performance}
    \label{tab:lorenz96_50_Best_performance}
    
    \newcommand{\res}[2]{\shortstack{#1 \\ {\scriptsize (#2)}}}
    
    \resizebox{\columnwidth}{!}{%
    \begin{tabular}{llcccc}
        \toprule
        &  & \multicolumn{2}{c}{F = 10} & \multicolumn{2}{c}{F = 40}  \\
        \cmidrule(lr){3-4} \cmidrule(lr){5-6} 
        Penalty & Metric & T=500 & T=1000 & T=500 & T=1000 \\
        \midrule
        \multirow{2}{*}{Jacob-L1}
          & AUROC & \res{0.694}{0.007} & \res{0.774}{0.031} & \res{0.746}{0.032} & \res{0.941}{0.122} \\
          \addlinespace[0.5em]
          & AUPRC & \res{0.428}{0.009} & \res{0.551}{0.043} & \res{0.500}{0.041} & \res{0.889}{0.217} \\
        \m